# 04 — Baseline Modeling

Train a logistic regression classifier using spatial cross-validation.

**⚠️  CRITICAL:** Random splits are forbidden. All evaluation uses spatial CV with 4 latitude-band folds. See `CLAUDE.md` and `models/spatial_cv.py`.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

## 1. Class Imbalance — Report First

In [ ]:
from features.build import load_feature_matrix, FEATURE_COLUMNS, LABEL_COLUMN

df = load_feature_matrix()
n_pos = df[LABEL_COLUMN].sum()
n_total = len(df)
ratio = (n_total - n_pos) / n_pos

print("=" * 50)
print("CLASS IMBALANCE REPORT (required)")
print("=" * 50)
print(f"Positive cells:  {n_pos:,} ({100*n_pos/n_total:.2f}%)")
print(f"Negative cells:  {n_total-n_pos:,} ({100*(n_total-n_pos)/n_total:.2f}%)")
print(f"Imbalance ratio: 1:{ratio:.0f}")
print()
print("Using class_weight='balanced' in logistic regression.")
print("Reporting: Precision, Recall, F1, AUC-ROC — NOT accuracy.")

## 2. Spatial CV Fold Layout

In [ ]:
from models.spatial_cv import SpatialLatitudeFoldCV
from visualization.hotspot_map import plot_cv_fold_layout

cv = SpatialLatitudeFoldCV()
fold_summary = cv.describe_folds(df['lat_centre'].values)
print("Fold composition:")
display(fold_summary)

fig = plot_cv_fold_layout(df)
plt.show()

## 3. Train with Spatial CV

In [ ]:
from models.train import train_with_spatial_cv, save_model

model, scaler, cv_results = train_with_spatial_cv(df)

print("\n=== CV Results ===")
display(cv_results[['fold', 'lat_band', 'precision', 'recall', 'f1', 'auc_roc', 'n_test', 'n_positive_test']])

numeric = ['precision', 'recall', 'f1', 'auc_roc']
print("\n=== Mean ± Std ===")
for col in numeric:
    vals = cv_results[col].dropna()
    print(f"{col:15s}: {vals.mean():.3f} ± {vals.std():.3f}")

## 4. Feature Importance

In [ ]:
import pandas as pd
coef_df = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#e74c3c' if c > 0 else '#3498db' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (standardised features)')
ax.set_title('Logistic Regression Feature Coefficients')
plt.tight_layout()
plt.show()
print("\nPositive = associated with higher P(hotspot)")
print("Coefficients are comparable because features are standardised.")

## 5. Prediction Surface

In [ ]:
from visualization.hotspot_map import plot_prediction_surface

X = df[FEATURE_COLUMNS].fillna(0).values
X_scaled = scaler.transform(X)
probs = model.predict_proba(X_scaled)[:, 1]

fig = plot_prediction_surface(df, probs)
plt.show()

## 6. Save Model

In [ ]:
save_model(model, scaler)
print("✅ Model and scaler saved.")

## Modeling Notes & Limitations

**Document before marking complete:**

- CV mean F1: ___
- CV mean AUC-ROC: ___
- Most important feature: ___
- Fold with worst performance: ___
- Likely reason for performance variation across folds: ___

**Limitations to carry forward:**
1. Observational bias in training labels (catalog coverage ≠ true distribution)
2. Synthetic tidal proxy if real data not available
3. Single model class (logistic regression) — explore alternatives only if baseline is well understood
4. Performance should be interpreted relative to the 1:N class imbalance (see imbalance report above)